# BMC : Low level functionality tutorial

In this notebook we will be demonstrating the individual functions from the BMC library that perform the heavy lifting of the geospatial data processing. The main focus of this ipynb will be to 
* demonstrate the functionality of the ``spatiotemporal_cube`` class

# ``spatiotemoral_cube``

The ``spatiotemoral_cube`` represents an abstract base class from which all the other cube classes inherit all their base functionality. This parent class contains the following general functionalities
* General parsing/formatting functions
* Data lake generation functionality
* Cubing generation functionality

Each of the children of this class must implement a strict set of abstract methods that act as "Translators." These translators take vendor-specific data structures (like CHELSA climate models or WEkEO API responses) and convert them into standard instructions. Once translated, the spatiotemporal_cube base class takes over, acting as a highly optimized "factory floor" to perform all the heavy I/O, mathematical grid snapping, memory chunking, and multithreading.

By separating the "What" (the child class) from the "How" (the base class), the BMC architecture ensures that your spatial physics are written exactly once and remain perfectly consistent across all environmental datasets.

## Master Grid Registry

Before doing any spatial processing, the engine needs to know the exact mathematical bounds of the target space. The spatiotemporal_cube contains a built-in GRID_REGISTRY—a dictionary defining strict coordinate reference systems (CRS) and bounding boxes to ensure flawless pixel-to-pixel alignment across distinct datasets.

In [1]:
from bmc.cube.spatiotemporal import spatiotemporal_cube

# Because it is an Abstract Base Class (ABC), we cannot instantiate it directly.
# Instead, we will inspect its class-level attributes, or create a dummy child class.
print("Available Master Grids:")
for grid_key, grid_info in spatiotemporal_cube.GRID_REGISTRY.items():
    print(f" - {grid_key}: {grid_info['crs']} | Res: {grid_info['resolution']}")

Available Master Grids:
 - EEA_100m: EPSG:3035 | Res: 100
 - EEA_250m: EPSG:3035 | Res: 250
 - EEA_500m: EPSG:3035 | Res: 500
 - EEA_1km: EPSG:3035 | Res: 1000
 - EEA_10km: EPSG:3035 | Res: 10000
 - Global_EqualArea_100m: EPSG:6933 | Res: 100
 - Global_EqualArea_250m: EPSG:6933 | Res: 250
 - Global_EqualArea_500m: EPSG:6933 | Res: 500
 - Global_EqualArea_1km: EPSG:6933 | Res: 1000
 - Global_EqualArea_10km: EPSG:6933 | Res: 10000
 - Global_WGS84_3sec: EPSG:4326 | Res: 0.0008333333333333333
 - Global_WGS84_7_5sec: EPSG:4326 | Res: 0.0020833333333333333
 - Global_WGS84_15sec: EPSG:4326 | Res: 0.004166666666666667
 - Global_WGS84_30sec: EPSG:4326 | Res: 0.008333333333333333
 - Global_WGS84_5min: EPSG:4326 | Res: 0.08333333333333333


![Visualisation of the master grids available in the BMC soatiotemporal_cube base class](img/grid_extents.png)

## Resampling strategy

The spatiotemporal base class serves as the mathematical framekwork that drives the resampling and reprojection of geospatial data to a harmonized data product. The underlying library that drives the math is the GDAL library, and as such we are restricted to the available resamplers defined within

In [12]:
from bmc.cube.spatiotemporal import spatiotemporal_cube

print("Available GDAL Resamplers in the Engine:")
for resampler_key in spatiotemporal_cube._GDAL_RESAMPLERS.keys():
    print(f" - {resampler_key}")

Available GDAL Resamplers in the Engine:
 - nearestNeighbour
 - bilinear
 - cubic
 - cubicSpline
 - lanczos
 - average
 - mode
 - max
 - min
 - med
 - q1
 - q3
 - sum
 - rms


### Categorical & Discrete Data
Use these for data like Land Cover or Forest Types to avoid creating decimal values that don't represent real categories:

* **nearestNeighbour**: Assigns the value of the single closest source pixel, preserving original discrete values without interpolation.
* **mode**: Assigns the most frequently occurring value among contributing pixels. The mathematical standard for downsampling categorical data.

---

### Continuous Data Smoothing
Use these for interpolating continuous gradients like Elevation or Temperature:

* **bilinear**: Distance-weighted average of the 4 closest source pixels.
* **cubic**: Distance-weighted cubic polynomial curve over the 16 nearest pixels.
* **cubicSpline**: 2D B-spline mathematical function over the 16 nearest pixels. Heavily smooths data and prevents "overshoot" (Runge's phenomenon). The gold standard for realistic, continuous gradients.
* **lanczos**: Complex windowed sinc function over the 36 nearest source pixels. Preserves high-frequency details and sharpness.

---

### Continuous Data Statistical Aggregation (Downsampling)
Use these when aggregating high-resolution continuous data into larger target grid cells:

* **average**: Arithmetic mean of all valid intersecting source pixels.
* **max / min**: Highest or lowest data value within the target footprint.
* **med**: Exact middle value (50th percentile) of contributing pixels.
* **q1 / q3**: First (25th) or third (75th) quartile of contributing pixels.
* **sum**: Addition of all valid intersecting source pixels.
* **rms**: Root Mean Square (quadratic mean). Emphasizes higher magnitude values.

## Behind the Scenes: Core Utility Functions
The `spatiotemporal_cube` class relies on several hidden, internal helper functions (often denoted by a leading underscore `_`) to safely manage execution state, validate spatial configurations, and handle mathematical conversions behind the scenes. 

Here is an overview of the engine's background mechanics:

### 1. Automated Execution Logging (`_setup_pipeline_logger`)
This private initialization function acts as the diagnostic brain of the data cube. 
* **File Streaming:** It automatically sets up a standard Python logger that streams all execution progress and non-fatal warnings directly to a `.log` file.
* **Global Safety Net:** Crucially, it binds a custom exception handler to Python's default error hook (`sys.excepthook`). If the pipeline encounters an unexpected bug or crash, this ensures the full traceback is permanently recorded as a "CRITICAL" issue before the script terminates.

### 2. Spatial Grid Validation (`resolve_grid_registry_key`)
Before any data is downloaded or warped, the engine must ensure the requested spatial framework actually exists. 
* **Dynamic Construction:** This function builds a master grid key by combining the user's target grid (e.g., "EEA") and resolution (e.g., "100m"). 
* **Strict Validation:** It checks this constructed key against the class's master `GRID_REGISTRY`. If a user requests an unsupported grid, it halts execution immediately and logs a spatial configuration error, preventing misaligned data.

### 3. Resolution Standardization (`_parse_res_to_meters`)
Raw ecological datasets come in wildly different formats (e.g., some use kilometers, some use meters). 
* **Mathematical Conversion:** This helper function strips strings (like '1km' or '10m') and mathematically converts them into standard float values in meters. 
* **Enabling Logic:** This standardization is what allows the engine to accurately sort and compare different available raw data resolutions during automated fetching.

### 4. Smart Strategy Routing (`_resolve_query_resolution`)
When users don't know the exact resolution of a remote dataset, they can pass a strategy (like 'highest' or 'lowest'). 
* **Dynamic Sorting:** This function maps the available inventory resolutions using the meter conversion helper, allowing it to easily pick the smallest distance for the 'highest' strategy, or the largest for the 'lowest'.
* **Safe Fallbacks:** If a user requests a highly specific resolution that does not exist in the vendor's catalog, this function logs a warning and automatically falls back to fetching the highest available resolution to keep the pipeline moving.

## Geospatial processing functionality